In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QUICK-PDE: Qiskit Function od ColibriTD
*Viz [referenci API](https://docs.quantum.ibm.com/api/functions/colibritd-pde)*

> **Note:** Qiskit Functions jsou experimentální funkce dostupné uživatelům IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (prostřednictvím IBM Quantum Platform API) Plan. Jsou ve stavu náhledového vydání a mohou se změnit.
## Přehled
Řešič parciálních diferenciálních rovnic (PDE) představený zde je součástí naší platformy Quantum Innovative Computing Kit (QUICK) (QUICK-PDE) a je zabalen jako Qiskit Function. S funkcí QUICK-PDE můžeš řešit doménově specifické parciální diferenciální rovnice na QPU IBM Quantum. Tato funkce je založena na algoritmu popsaném v [dokumentu H-DES od ColibriTD](https://arxiv.org/abs/2410.01130). Tento algoritmus dokáže řešit složité multifyzikální problémy, počínaje výpočetní dynamikou tekutin (CFD) a deformací materiálů (MD), přičemž další případy použití přibývají.

Při řešení diferenciálních rovnic jsou zkušební řešení zakódována jako lineární kombinace ortogonálních funkcí (typicky Čebyševovy polynomy, konkrétně $2^n$ z nich, kde $n$ je počet qubitů kódujících tvou funkci), parametrizované úhly Variable Quantum Circuit (VQC). Ansatz generuje stav kódující funkci, který je vyhodnocován pozorovatelnými, jejichž kombinace umožňují vyhodnotit funkci ve všech bodech. Poté můžeš vyhodnotit ztrátovou funkci, do které jsou zakódovány diferenciální rovnice, a doladit úhly v hybridní smyčce, jak je znázorněno níže. Zkušební řešení se postupně přibližují skutečným řešením, dokud nedosáhneš uspokojivého výsledku.

![Pracovní postup funkce QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

Kromě této hybridní smyčky můžeš také řetězit různé optimalizátory dohromady. To je užitečné, když chceš, aby globální optimalizátor nalezl dobrou sadu úhlů, a poté jemnější optimalizátor sledoval gradient k nejlepší sadě sousedních úhlů. V případě výpočetní dynamiky tekutin (CFD) výchozí optimalizační sekvence přináší nejlepší výsledky – v případě deformace materiálů (MD) sice výchozí nastavení poskytuje dobré výsledky, ale můžeš ho dále konfigurovat pro výhody specifické pro daný problém.

Poznámka: pro každou proměnnou funkce specifikujeme počet qubitů (se kterým si můžeš hrát). Naskládáním 10 identických Circuit a vyhodnocením 10 identických pozorovatelných na různých qubitech v rámci jednoho velkého Circuit můžeš provádět potlačení šumu v rámci procesu optimalizace CMA, spoléhajíce na metodu noise learner, a výrazně snížit počet potřebných měření.
### Výpočetní dynamika tekutin
Neviskózní Burgersova rovnice modeluje proudění neviskózních tekutin takto:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ představuje pole rychlosti tekutiny. Tento případ použití má časovou okrajovou podmínku: můžeš vybrat počáteční podmínku a pak nechat systém relaxovat. V současnosti jsou přijímány pouze lineární funkce jako počáteční podmínky: $ax + b$.

Argumenty diferenciálních rovnic pro CFD jsou na pevné mřížce, jak následuje:

- $t$ je v rozsahu 0 až 0,95 s 30 vzorkovacími body. $x$ je v rozsahu 0 až 0,95 s krokem 0,2375.

### Deformace materiálů
Tento případ použití se zaměřuje na hypoelastickou deformaci s jednorozměrným zkouškou tahem, při níž je tyč pevně uchycena v prostoru a táhnuta na druhém konci. Popis problému je následující:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ představuje objemový modul pružnosti roztahovaného materiálu, $n$ exponent mocninného zákona, $b$ sílu na jednotku hmotnosti, $\epsilon_0$ proporcionální mez napětí, $\sigma_0$ proporcionální mez přetvoření, $u$ funkci napětí a $\sigma$ funkci přetvoření.

Uvažovaná tyč má jednotkovou délku. Tento případ použití má okrajovou podmínku pro povrchové napětí $t$, tedy množství práce potřebné k natažení tyče.

Argumenty diferenciálních rovnic pro MD jsou na pevné mřížce, jak následuje:

- $x$ je v rozsahu 0 až 1 s krokem 0,04.

## Benchmarky
Následující tabulka uvádí statistiky různých spuštění naší funkce.

| Příklad                           | Počet qubitů | Inicializace          | Chyba     | Celkový čas (min) | Využití runtime (min) |
| --------------------------------- | ------------ | --------------------- | --------- | ----------------- | --------------------- |
| Neviskózní Burgersova rovnice     | 50           | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                | 25                    |
| Hypoelastická 1D zkouška tahem    | 18           | `RANDOM`              | $10^{-2}$ | 123               | 100                   |
## Začínáme
Vyplň [formulář pro vyžádání přístupu k funkci QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK). Poté, za předpokladu, že sis již [uložil svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do lokálního prostředí, vyber funkci takto:

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_cloud / ibm_quantum_platform",
    instance="USER_CRN / HGP",
    token="USER_API_KEY / IQP_API_TOKEN",
)

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

In [ ]:
quick = catalog.load("colibritd/quick-pde")

Zkontroluj [stav](/guides/functions#check-job-status) úlohy své Qiskit Function nebo získej [výsledky](/guides/functions#retrieve-results) takto:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(
    use_case="CFD_BURGER", physical_parameters={"a": 1.0, "b": 0.0}
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [ ]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())
solution = job.result()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_result_3d(result):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    t, x = np.meshgrid(result["samples"]["t"], result["samples"]["x"])

    ax.plot_surface(
        t,
        x,
        result["functions"]["u"],
        edgecolor="royalblue",
        lw=0.25,
        rstride=26,
        cstride=26,
        alpha=0.3,
    )
    ax.scatter(t, x, result["functions"]["u"], marker=".")
    ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

    plt.show()


# Call
plot_result_3d(solution)

![Výstup předchozí buňky kódu](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Deformace materiálů
Případ použití deformace materiálů vyžaduje fyzikální parametry tvého materiálu a přiloženou sílu, jak následuje:

In [ ]:
# Launches the solving for an arbitrary mu
job = quick.run(use_case="CFD_EULER", physical_parameters={"mu": 0.1})

solution = job.result()


# Colorplot function
def plot_result_2d(result):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    configs = {
        "g": {"cmap": "viridis", "title": "g(t, x)"},
        "u": {"cmap": "plasma", "title": "u(t, x)"},
    }

    t = result["samples"]["t"]
    x = result["samples"]["x"]

    for ax, (field, cfg) in zip(axes, configs.items()):
        v = result["functions"][field]

        im = ax.contourf(t, x, v, levels=50, cmap=cfg["cmap"])
        fig.colorbar(im, ax=ax, label=cfg["title"])

        ax.set_xlabel("t")
        ax.set_ylabel("x")
        ax.set_title(cfg["title"], fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()


plot_result_2d(solution)

![Výstup předchozí buňky kódu](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

Následuje příklad, jak získat hodnotu funkce pro konkrétní sadu souřadnic:

In [ ]:
# Select the properties of your material
job = quick.run(
    use_case="MD",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# Plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

## Načítání chybových zpráv
Pokud je stav tvé úlohy `ERROR`, použij `job.error_message()` k načtení chybové zprávy pro pomoc při ladění, jak následuje:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Fetch error messages

If your workload status is `ERROR`, use `job.error_message()` to fetch the error message to help debug, as follows:

In [ ]:
job = quick.run(use_case="MD", physical_params={})

print(job.error_message())


# A wrapper can also be used for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

## Získat podporu

Pro podporu kontaktuj qiskit-function-support@colibritd.com.

## Další kroky

> **Tip:** - Vyplň formulář pro [vyžádání přístupu k funkci QUICK-PDE](https://forms.cloud.microsoft/e/3Wi9cbjQPK).
> - Navštiv [referenci API](https://docs.quantum.ibm.com/api/functions/colibritd-pde) pro tuto Qiskit Function.
> - Vyzkoušej modelování proudění neviskózní tekutiny pomocí QUICK-PDE v [tutoriálu](/tutorials/colibritd-pde).
> - Prostuduj [Jaffali, H., et al. (2025). H-DES: a Quantum-Classical Hybrid Differential Equation Solver. arXiv preprint arXiv:2410.01130](https://arxiv.org/abs/2410.01130).